# Example 19 — Womersley flow: pulsatile channel (the biofluids classic)

Blood flow in arteries is driven by an oscillating pressure gradient. In a 2-D channel:
$$u_t = G\cos(\omega t) + \nu\,u_{yy},\qquad u(\pm 1, t) = 0,$$
with exact (complex-variable) solution
$$u = \Re\left[\frac{G}{i\omega}\Big(1 - \frac{\cosh\lambda y}{\cosh\lambda}\Big)e^{i\omega t}\right],\qquad \lambda = \sqrt{i\omega/\nu}.$$
The **Womersley number** $Wo = h\sqrt{\omega/\nu}$ (here 5) controls the character: low
$Wo$ → quasi-steady parabolas; high $Wo$ → plug flow with **annular overshoot** near the
walls and phase lag in the core.

**PINN design:** hard no-slip via $u = (1-y^2)\,N(y, \sin\omega t, \cos\omega t)$ and
hard-periodic time (Example 17's trick) → the loss is **pure residual**, nothing else.

Verified: L2 ≈ 8.0e-03–1.7e-02 across four phases, ~54 s on CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

NU, WO = 1.0, 5.0
W = WO**2          # omega, since h = 1
G = W              # forcing amplitude scaled for O(1) velocities
LAM = np.sqrt(1j*W/NU)
def u_exact(y, t):
    c = (G/(1j*W))*(1 - np.cosh(LAM*y)/np.cosh(LAM))*np.exp(1j*W*t)
    return np.real(c)

net = nn.Sequential(nn.Linear(3,48), nn.Tanh(), nn.Linear(48,48), nn.Tanh(),
                    nn.Linear(48,48), nn.Tanh(), nn.Linear(48,1)).to(device)
u_of = lambda y, t: (1-y**2)*net(torch.cat([y, torch.sin(W*t), torch.cos(W*t)], 1))
opt = torch.optim.Adam(net.parameters(), 2e-3)

t0 = time.perf_counter()
for e in range(5000):
    opt.zero_grad()
    y = (torch.rand(2000,1,device=device)*2-1).requires_grad_(True)
    t = (torch.rand(2000,1,device=device)*(2*np.pi/W)).requires_grad_(True)
    u = u_of(y, t)
    res = g1(u,t) - G*torch.cos(W*t) - NU*g1(g1(u,y),y)
    (res**2).mean().backward(); opt.step()      # pure residual — all BCs are hard
if device.type=='cuda': torch.cuda.synchronize()
print(f'training: {time.perf_counter()-t0:.0f} s   (Wo = {WO})')

yg = np.linspace(-1, 1, 200)
yt = torch.tensor(yg, dtype=torch.float32, device=device).reshape(-1,1)
plt.figure(figsize=(8.5,4.5))
for ph, c in zip((0, 0.25, 0.5, 0.75), ('tab:blue','tab:orange','tab:green','tab:red')):
    tv = ph*2*np.pi/W
    with torch.no_grad(): up = u_of(yt, torch.full_like(yt, tv)).cpu().numpy().ravel()
    plt.plot(u_exact(yg, tv), yg, c, lw=2, alpha=.5)
    plt.plot(up, yg, '--', color=c, lw=1.3, label=f'phase {ph}T')
plt.xlabel('u'); plt.ylabel('y'); plt.legend(fontsize=9); plt.grid(alpha=.3)
plt.title(f'Womersley profiles, Wo={WO}: plug core + annular overshoot (solid=exact, dashed=PINN)')
plt.tight_layout(); plt.show()

## Observations
- **Annular effect, visible:** at $Wo=5$ the velocity peaks *near the walls*, not the
  centre — the core's inertia can't keep up with the forcing, the thin wall layers can.
  The reverse of everything steady-flow intuition says.
- **All constraints hard** — $(1-y^2)$ kills no-slip, $(\sin,\cos\omega t)$ kills
  transients — leaving a single-term loss. The cleanest advanced example in the kit.
- **Phase lag:** core velocity lags the pressure gradient by nearly 90° at high $Wo$ (an
  RL-circuit analogy: inertia = inductance).
- **Why biofluids cares:** $Wo\approx$ 10–20 in the aorta, ~3 in coronaries — this one
  dimensionless number decides the waveform shape.

**Try:** make $Wo$ a network input and sweep 1→10 in one training (Example 5's surrogate);
compare the $Wo=1$ limit against Poiseuille (Example 12).